In [1]:
import sqlite3, pandas as pd
from pathlib import Path

ROOT = Path("/home/py/groundwater/")
DATA = ROOT / 'data'
GOWN = DATA / 'gown'
con = sqlite3.connect(DATA / "gw.sqlite")

In [2]:
bow_valley = [931, 301, 303, 386, 305, 764, 760, 759, 364]

In [3]:
meta = pd.read_csv(GOWN / 'tables' / 'stations_meta.csv')
meta['groundwater.Production_S']

0         157 - 163.4
1         20.5 - 23.5
2       4.3 - 5.8/6.4
3           64 - 72.5
4         140 - 144.8
            ...      
1153      31.4 - 32.9
1154       94.8 - 125
1155        106 - 110
1156              NaN
1157    338.3 - 350.3
Name: groundwater.Production_S, Length: 1158, dtype: object

In [4]:
meta = pd.read_csv(GOWN / 'tables' / 'stations_meta.csv')
meta = meta[['station_no', 'groundwater.StationCode_S', 'groundwater.GICWellID_S']]
meta[meta['groundwater.StationCode_S'].isin(bow_valley)]

,station_no,groundwater.StationCode_S,groundwater.GICWellID_S
12,05BFG006,301,370217.0
23,05BFG004,303,370209.0
36,05BFG005,305,370221.0
79,05BEG021,764,1021710.0
94,05BEG023,364,370213.0
104,05BEG020,759,496371.0
121,05BFG003,386,370218.0
148,05BFG001,931,404368.0
196,05BEG018,760,496372.0


In [5]:
well_ids = meta[meta['groundwater.StationCode_S'].isin(bow_valley)]['groundwater.GICWellID_S'].values

In [6]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)['name']
for t in tables:
    n = pd.read_sql(f'SELECT COUNT(*) n FROM "{t}"', con)['n'][0]
    print(f"{t:40s} {n:>12,} rows")

Analysis_Items                              2,123,638 rows
Chemical_Analysis                             115,709 rows
Driller_Drilling_Company                        2,428 rows
Drillers                                          689 rows
Drilling_Companies                              1,659 rows
Elements                                           65 rows
Geophysical_Logs                              782,457 rows
Lithologies                                 2,619,656 rows
Other_Seals                                    42,663 rows
Perforations                                  156,229 rows
Pump_Test_Items                             1,857,676 rows
Pump_Tests                                    299,246 rows
Screens                                        38,627 rows
SwitchboardItems                                    7 rows
Well_Owners                                   461,692 rows
Well_Reports                                  457,743 rows
Wells                                         452,923 ro

In [19]:
pd.read_sql('PRAGMA table_info("Geophysical_Logs")', con)

,cid,name,type,notnull,dflt_value,pk
0,0,Geophysical_Log_ID,INTEGER,0,None,0
1,1,Well_Report_ID,INTEGER,0,None,0
2,2,Log_Type,varchar,0,None,0
3,3,Log_Taken_Flag,INTEGER,1,None,0
4,4,Sent_to_AENV_Flag,INTEGER,1,None,0


In [8]:
pd.read_sql('SELECT * FROM "Analysis_Items" LIMIT 20', con)

,Element_Name,Element_Symbol,Decimal_Places,Value,Chemical_Analysis_ID
0,Total Dissolved Solids,TDS,0,2606,2000001
1,Total Dissolved Solids,TDS,0,6865,2000002
2,Total Dissolved Solids,TDS,0,4726,2000003
3,Total Dissolved Solids,TDS,0,2074,2000004
4,Total Dissolved Solids,TDS,0,3776,2000005
5,Total Dissolved Solids,TDS,0,524,2000006
6,Total Dissolved Solids,TDS,0,672,2000007
7,Total Dissolved Solids,TDS,0,858,2000008
8,Total Dissolved Solids,TDS,0,1016,2000009
9,Total Dissolved Solids,TDS,0,870,2000010


In [9]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)["name"].tolist()

cols = {t: pd.read_sql(f'PRAGMA table_info("{t}")', con)["name"].tolist() for t in tables}

key = "GIC_Well_ID"
direct = [t for t in tables if key in cols[t]]
indirect = [t for t in tables if key not in cols[t]]
print("direct:", direct)
print("no GIC_Well_ID:", indirect)

direct: ['Lithologies', 'Screens', 'Wells']
no GIC_Well_ID: ['Analysis_Items', 'Chemical_Analysis', 'Driller_Drilling_Company', 'Drillers', 'Drilling_Companies', 'Elements', 'Geophysical_Logs', 'Other_Seals', 'Perforations', 'Pump_Test_Items', 'Pump_Tests', 'SwitchboardItems', 'Well_Owners', 'Well_Reports', 'Boreholes']


In [10]:
ph = ",".join("?" * len(well_ids))

data = {}
for t in direct:
    data[t] = pd.read_sql(f'SELECT * FROM "{t}" WHERE "{key}" IN ({ph})', con, params=well_ids)

for t, df in data.items():
    print(f"{t:35s} {len(df):>8,} rows")

Lithologies                               70 rows
Screens                                    3 rows
Wells                                      9 rows


## Full extraction for `well_ids`

Only `Wells`, `Lithologies` and `Screens` carry `GIC_Well_ID`, so the other tables have to be
reached through the key chain:

```
GIC_Well_ID -> Wells.Well_ID -> Well_Reports.Well_Report_ID -> Pump_Tests.Pump_Test_ID
                            |                              -> Chemical_Analysis.Chemical_Analysis_ID
```

| via | tables |
|---|---|
| `GIC_Well_ID` | Wells |
| `Well_ID` | Well_Reports, Well_Owners, Chemical_Analysis |
| `Well_Report_ID` | Lithologies, Screens, Boreholes, Perforations, Other_Seals, Geophysical_Logs, Pump_Tests |
| `Pump_Test_ID` | Pump_Test_Items |
| `Chemical_Analysis_ID` | Analysis_Items |
| lookup | Drillers, Drilling_Companies |

Two gotchas that make the `GIC_Well_ID IN (...)` shortcut unsafe in general:

- **`Well_ID` is not always `GIC_Well_ID`.** They match for 8 of these 9 wells, but GIC `1021710`
  (05BEG021) is `Well_ID` `11557173`. Anything joined on the GIC value alone silently drops it.
- **A well can have several reports.** 370209 and 370221 each carry extra 2012 *Old Well-Yield*
  reports on top of the original 1960s drilling record, and those reports own their own pump tests.

`Elements` and `SwitchboardItems` are excluded — a units lookup and leftover MS Access UI cruft,
neither is well-linked.

In [11]:
def fetch(table, col, ids):
    """All rows of `table` whose `col` is in `ids` (deduped, NaN-dropped, cast to int)."""
    ids = [int(i) for i in pd.unique(pd.Series(list(ids)).dropna())]
    if not ids:
        return pd.read_sql(f'SELECT * FROM "{table}" WHERE 0', con)
    q = f'SELECT * FROM "{table}" WHERE "{col}" IN ({",".join("?" * len(ids))})'
    return pd.read_sql(q, con, params=ids)


# station_no <-> station_code <-> GIC_Well_ID <-> Well_ID
xwalk = (meta[meta['groundwater.GICWellID_S'].isin(well_ids)]
         .rename(columns={'groundwater.StationCode_S': 'station_code',
                          'groundwater.GICWellID_S': 'GIC_Well_ID'})
         .astype({'GIC_Well_ID': 'int64'}))

wells = fetch('Wells', 'GIC_Well_ID', xwalk.GIC_Well_ID)
xwalk = xwalk.merge(wells[['GIC_Well_ID', 'Well_ID', 'GOA_Well_Tag_Number']], on='GIC_Well_ID', how='left')
reports = fetch('Well_Reports', 'Well_ID', xwalk.Well_ID)

gic_of_well = dict(zip(xwalk.Well_ID, xwalk.GIC_Well_ID))
stn_of_well = dict(zip(xwalk.Well_ID, xwalk.station_no))
well_of_report = dict(zip(reports.Well_Report_ID, reports.Well_ID))


def tag(df, id_col, to_well):
    """Prepend station_no / GIC_Well_ID so every extracted table is self-describing."""
    if df.empty:
        return df
    w = df[id_col].map(to_well)
    out = df.drop(columns=['station_no', 'GIC_Well_ID'], errors='ignore')
    out.insert(0, 'GIC_Well_ID', w.map(gic_of_well).values)
    out.insert(0, 'station_no', w.map(stn_of_well).values)
    return out


xwalk[['station_no', 'station_code', 'GIC_Well_ID', 'Well_ID']]

,station_no,station_code,GIC_Well_ID,Well_ID
0,05BFG006,301,370217,370217
1,05BFG004,303,370209,370209
2,05BFG005,305,370221,370221
3,05BEG021,764,1021710,11557173
4,05BEG023,364,370213,370213
5,05BEG020,759,496371,496371
6,05BFG003,386,370218,370218
7,05BFG001,931,404368,404368
8,05BEG018,760,496372,496372


In [12]:
identity = {w: w for w in xwalk.Well_ID}

data = {}
data['Wells'] = tag(wells, 'Well_ID', identity)
data['Well_Reports'] = tag(reports, 'Well_ID', identity)
data['Well_Owners'] = tag(fetch('Well_Owners', 'Well_ID', xwalk.Well_ID), 'Well_ID', identity)
data['Chemical_Analysis'] = tag(fetch('Chemical_Analysis', 'Well_ID', xwalk.Well_ID), 'Well_ID', identity)

for t in ['Lithologies', 'Screens', 'Boreholes', 'Perforations',
          'Other_Seals', 'Geophysical_Logs', 'Pump_Tests']:
    data[t] = tag(fetch(t, 'Well_Report_ID', reports.Well_Report_ID), 'Well_Report_ID', well_of_report)

# grandchildren, keyed off their parent's id
pt = data['Pump_Tests']
data['Pump_Test_Items'] = tag(
    fetch('Pump_Test_Items', 'Pump_Test_ID', pt.Pump_Test_ID),
    'Pump_Test_ID', dict(zip(pt.Pump_Test_ID, pt.Well_Report_ID.map(well_of_report))))

ca = data['Chemical_Analysis']
data['Analysis_Items'] = tag(
    fetch('Analysis_Items', 'Chemical_Analysis_ID', ca.Chemical_Analysis_ID),
    'Chemical_Analysis_ID', dict(zip(ca.Chemical_Analysis_ID, ca.Well_ID)))

# lookups: many-to-many with wells, so left untagged
data['Drillers'] = fetch('Drillers', 'Driller_ID', reports.Driller_ID)
data['Drilling_Companies'] = fetch('Drilling_Companies', 'Drilling_Company_ID',
                                   pd.concat([reports.Drilling_Company_ID, wells.Drilling_Company_ID]))

for t, df in data.items():
    n_wells = df['GIC_Well_ID'].nunique() if 'GIC_Well_ID' in df else 0
    print(f"{t:22s} {len(df):>6,} rows   {n_wells:>2}/9 wells")

Wells                       9 rows    9/9 wells
Well_Reports               14 rows    9/9 wells
Well_Owners                 8 rows    8/9 wells
Chemical_Analysis           8 rows    3/9 wells
Lithologies                70 rows    9/9 wells
Screens                     3 rows    3/9 wells
Boreholes                   8 rows    8/9 wells
Perforations                2 rows    2/9 wells
Other_Seals                 0 rows    0/9 wells
Geophysical_Logs            8 rows    6/9 wells
Pump_Tests                 22 rows    7/9 wells
Pump_Test_Items           283 rows    6/9 wells
Analysis_Items             98 rows    3/9 wells
Drillers                    3 rows    0/9 wells
Drilling_Companies          7 rows    0/9 wells


In [13]:
# The construction report = earliest report that actually records drilling. The 2012
# "Old Well-Yield" reports on 370209/370221 stay in Well_Reports but must not be mixed
# into the construction summary (a groupby().last() would coalesce columns across rows).
rep = data['Well_Reports'].copy()
rep['_rank'] = rep.Drilling_End_Date.isna().astype(int)
build = (rep.sort_values(['_rank', 'Drilling_End_Date', 'Well_Report_ID'])
            .groupby('GIC_Well_ID').head(1).set_index('GIC_Well_ID'))

counts = pd.DataFrame({
    f'n_{name}': data[t].groupby('GIC_Well_ID').size()
    for name, t in [('reports', 'Well_Reports'), ('lithology', 'Lithologies'),
                    ('screens', 'Screens'), ('boreholes', 'Boreholes'),
                    ('perforations', 'Perforations'), ('geophys_logs', 'Geophysical_Logs'),
                    ('pump_tests', 'Pump_Tests'), ('pump_readings', 'Pump_Test_Items'),
                    ('chem_samples', 'Chemical_Analysis'), ('chem_analytes', 'Analysis_Items')]
}).reindex(xwalk.GIC_Well_ID).fillna(0).astype(int)

summary = (xwalk.set_index('GIC_Well_ID')[['station_no', 'station_code', 'Well_ID', 'GOA_Well_Tag_Number']]
           .join(data['Wells'].set_index('GIC_Well_ID')[
               ['Latitude', 'Longitude', 'Elevation', 'LSD', 'Section', 'Township', 'Range', 'Meridian']])
           .join(build[['Type_of_Work', 'Well_Use', 'Drilling_Method', 'Drilling_End_Date',
                        'Total_Depth_Drilled', 'Casing_Material', 'Casing_OD', 'Casing_Bottom',
                        'Recommended_Rate', 'Recommended_Intake_Depth']])
           .join(counts)
           .sort_values('station_code'))

summary

,station_no,station_code,Well_ID,GOA_Well_Tag_Number,Latitude,Longitude,Elevation,LSD,Section,Township,...,n_reports,n_lithology,n_screens,n_boreholes,n_perforations,n_geophys_logs,n_pump_tests,n_pump_readings,n_chem_samples,n_chem_analytes
GIC_Well_ID,,,,,,,,,,,,,,,,,,,,,
370217,05BFG006,301,370217,None,50.950108,-115.153984,5290.869400,14,11,23,...,1,8,0,1,0,1,3,67,1,8
370209,05BFG004,303,370209,None,50.956421,-115.157904,5474.993400,6,14,23,...,4,20,0,1,0,1,5,73,2,27
370221,05BFG005,305,370221,None,50.975848,-115.180297,6788.159449,11,22,23,...,3,6,0,1,0,1,6,69,5,63
370213,05BEG023,364,370213,None,51.073672,-115.115353,4215.160800,5,30,24,...,1,7,1,1,0,0,0,0,0,0
370218,05BFG003,386,370218,None,50.957081,-115.182339,NaN,5,15,23,...,1,3,0,1,0,0,4,59,0,0
496371,05BEG020,759,496371,None,51.058939,-115.154195,4238.313700,6,23,24,...,1,9,1,1,0,2,1,6,0,0
496372,05BEG018,760,496372,None,51.107283,-115.366383,NaN,11,5,25,...,1,3,1,1,0,2,1,9,0,0
1021710,05BEG021,764,11557173,None,51.127674,-115.381133,4527.778871,2,18,25,...,1,9,0,0,1,1,2,0,0,0
404368,05BFG001,931,404368,None,50.884138,-115.140346,4955.098400,7,23,22,...,1,5,0,1,1,0,0,0,0,0


In [14]:
# OUT = DATA / 'bow_valley' / 'awwid'
# OUT.mkdir(parents=True, exist_ok=True)

# summary.to_csv(OUT / 'well_summary.csv')
# for t, df in data.items():
#     if len(df):
#         df.to_csv(OUT / f'{t.lower()}.csv', index=False)

# print(f"{OUT}:")
# for f in sorted(OUT.iterdir()):
#     print(f"  {f.name:28s} {f.stat().st_size:>8,} B")

## Cross-section inputs

Compiles the extracted AWWID tables + `stations_meta` + the GOWN water-level series into
`data/bow_valley/cross_section/`. The README written alongside them carries the full datum notes
and caveats.

**Units.** AWWID stores depths in *feet*, `stations_meta` in *metres*. Verified rather than
assumed: the deepest logged interval equals AWWID `Total_Depth_Drilled` exactly in all 9 wells,
and converting that to metres reproduces `meta.Depth_N` to within 0.04 m in 8 of 9 (05BEG021 is
the exception, 61.0 vs 59.1 m). Everything below is metres.

**Datum.** `meta.GWREF_DATUM` is the *top of casing*, not ground — `absval + mbtoc == GWREF_DATUM`
to 1e-4 m for 7 of the 9 wells. Using it as the collar keeps lithology, completions and water
levels mutually consistent. AWWID's `Wells.Elevation` is a coarser estimate (off by 9.5 m at
05BFG006) and is kept only as a cross-check column.

**Screens.** `meta.Production_S` carries a completion interval for all 9 wells, whereas AWWID
`Screens`/`Perforations` covers only 5. The two agree to within 0.04 m wherever both exist, so
`Production_S` is trustworthy for the remaining 4.

> `meta` was narrowed to 3 columns further up, so the full station table is re-read here.

In [15]:
import numpy as np
from pyproj import Transformer

AWWID = DATA / 'bow_valley' / 'awwid'
XS = DATA / 'bow_valley' / 'cross_section'
XS.mkdir(parents=True, exist_ok=True)
FT = 0.3048

aw = {f.stem: pd.read_csv(f) for f in AWWID.glob('*.csv')}
sm = aw['well_summary'].set_index('station_no')

meta_full = pd.read_csv(GOWN / 'tables' / 'stations_meta.csv')
meta_full = meta_full[meta_full['groundwater.StationCode_S'].isin(bow_valley)].copy()
meta_full['code'] = meta_full['groundwater.StationCode_S'].astype(str).str.zfill(4)

# ---------------------------------------------------------------- collars ----
tr = Transformer.from_crs('EPSG:4326', 'EPSG:32611', always_xy=True)
c = meta_full[['station_no', 'code', 'groundwater.StationCode_S', 'groundwater.GICWellID_S',
               'station_latitude', 'station_longitude', 'GWREF_DATUM', 'groundwater.Depth_N',
               'groundwater.Aquifer_S', 'groundwater.Lithology_S', 'groundwater.AquiferType_K',
               'groundwater.Production_S', 'groundwater.DrillDate_D', 'general.LSD_S',
               'general.Section_S', 'general.Township_S', 'general.Range_S',
               'general.Meridian_S']].copy()
c.columns = ['station_no', 'code', 'station_code', 'gic_well_id', 'latitude', 'longitude',
             'mp_elev_m', 'depth_str', 'aquifer', 'aquifer_lithology', 'aquifer_type',
             'production_str', 'drill_date', 'lsd', 'section', 'township', 'range', 'meridian']
c['gic_well_id'] = c.gic_well_id.astype('int64')
c['x_utm11n'], c['y_utm11n'] = tr.transform(c.longitude.values, c.latitude.values)
c['well_depth_m'] = c.depth_str.str.replace(' m', '', regex=False).astype(float)

c = c.set_index('station_no')
c['well_id'] = sm.Well_ID
c['td_m'] = (sm.Total_Depth_Drilled * FT).round(2)
c['awwid_elev_m'] = (sm.Elevation * FT).round(2)           # coarse, cross-check only
c['elev_discrepancy_m'] = (c.awwid_elev_m - c.mp_elev_m).round(2)
c['type_of_work'] = sm.Type_of_Work
c['well_use'] = sm.Well_Use
c['drilling_method'] = sm.Drilling_Method
c['casing_material'] = sm.Casing_Material.replace('Unknown', np.nan)
c['casing_od_in'] = sm.Casing_OD.replace(0, np.nan)
c['casing_bottom_m'] = (sm.Casing_Bottom.replace(0, np.nan) * FT).round(2)
c['mp_base_elev_m'] = (c.mp_elev_m - c.td_m).round(2)      # borehole base, MP datum
c = c.reset_index()

c[['station_no', 'code', 'mp_elev_m', 'well_depth_m', 'aquifer', 'aquifer_type',
   'elev_discrepancy_m', 'x_utm11n', 'y_utm11n']].round(1)

,station_no,code,mp_elev_m,well_depth_m,aquifer,aquifer_type,elev_discrepancy_m,x_utm11n,y_utm11n
0,05BFG006,0301,1603.1,12.2,Rocky Mountain,---,9.5,629672.4,5645897.0
1,05BFG004,0303,1668.0,36.6,Rocky Mountain,---,0.7,629375.6,5646594.5
2,05BFG005,0305,2067.5,11.6,Fernie,---,1.5,627749.1,5648714.8
3,05BEG021,0764,1378.1,59.1,Surficial,---,2.0,613280.0,5665269.0
4,05BEG023,0364,1284.6,33.8,Buried Valley,Confined,0.2,632030.3,5659707.4
5,05BEG020,0759,1291.8,219.5,Calgary Valley,Confined,0.0,629350.7,5657999.9
6,05BFG003,0386,1640.0,12.8,Surficial,---,NaN,627657.8,5646625.1
7,05BFG001,0931,1509.8,30.5,Surficial,Unconfined,0.5,630811.9,5638588.1
8,05BEG018,0760,1315.5,85.3,Calgary Valley,Unconfined,NaN,614358.4,5663027.6


In [16]:
# Lithologies.Depth is the interval BASE, so intervals are built by chaining depths from 0.
lith = aw['lithologies']
lith = lith[lith.Material != 'Old Well']           # 2012 stub reports, not a real log

rep = aw['well_reports'].copy()                    # same construction-report rule as the summary
rep['_rank'] = rep.Drilling_End_Date.isna().astype(int)
build_report = (rep.sort_values(['_rank', 'Drilling_End_Date', 'Well_Report_ID'])
                   .groupby('station_no').head(1).set_index('station_no').Well_Report_ID.to_dict())
lith = lith[[r.Well_Report_ID == build_report[r.station_no] for r in lith.itertuples()]]

CLASS = {
    'Gravel': 'coarse granular', 'Sand & Gravel': 'coarse granular',
    'Gravel & Boulders': 'coarse granular', 'Sand': 'coarse granular',
    'Till': 'till/diamict', 'Till & Rocks': 'till/diamict', 'Rocks': 'till/diamict',
    'Clay': 'fine grained', 'Clay & Gravel': 'fine grained', 'Clay & Sand': 'fine grained',
    'Shale': 'bedrock', 'Sandstone': 'bedrock', 'Coal': 'bedrock',
    'Bedrock': 'bedrock', 'Limestone': 'bedrock',
    'Shale & Gravel': 'bedrock (mixed)', 'Sandstone & Gravel': 'bedrock (mixed)',
}

recs = []
for stn, g in lith.groupby('station_no'):
    g = g.sort_values('Depth')                     # 05BEG020 is stored out of depth sequence
    top = 0.0
    for r in g.itertuples():
        base = r.Depth * FT
        recs.append({'station_no': stn, 'from_m': round(top, 2), 'to_m': round(base, 2),
                     'thickness_m': round(base - top, 2), 'material': r.Material,
                     'description': r.Description, 'colour': r.Colour,
                     'lith_class': CLASS.get(r.Material, 'other'),
                     'water_bearing': int(r.Water_Bearing),
                     'from_ft': round(top / FT), 'to_ft': r.Depth})
        top = base

li = pd.DataFrame(recs).merge(c[['station_no', 'code', 'mp_elev_m']], on='station_no')
li['top_elev_m'] = (li.mp_elev_m - li.from_m).round(2)
li['base_elev_m'] = (li.mp_elev_m - li.to_m).round(2)
li = li[['station_no', 'code', 'from_m', 'to_m', 'thickness_m', 'top_elev_m', 'base_elev_m',
         'material', 'description', 'colour', 'lith_class', 'water_bearing', 'from_ft', 'to_ft']]

# does the log reach total depth? (gap_m > 0 means an unlogged interval at the bottom)
reach = (li.groupby('station_no').to_m.max().rename('log_base_m').to_frame()
           .join(c.set_index('station_no')[['well_depth_m', 'td_m']]))
reach['gap_vs_meta_m'] = (reach.well_depth_m - reach.log_base_m).round(2)
print(reach.to_string(), '\n')
li.groupby(['station_no', 'lith_class']).thickness_m.sum().unstack(fill_value=0).round(1)

            log_base_m  well_depth_m    td_m  gap_vs_meta_m
station_no                                                 
05BEG018         85.34         85.30   85.34          -0.04
05BEG020        219.46        219.50  219.46           0.04
05BEG021         60.96         59.10   60.96          -1.86
05BEG023         33.83         33.80   33.83          -0.03
05BFG001         30.48         30.48   30.48           0.00
05BFG003         12.80         12.80   12.80           0.00
05BFG004         36.58         36.58   36.58           0.00
05BFG005         11.58         11.58   11.58           0.00
05BFG006         12.19         12.20   12.19           0.01 



lith_class,bedrock,bedrock (mixed),coarse granular,fine grained,till/diamict
station_no,,,,,
05BEG018,12.8,0.0,68.6,4.0,0.0
05BEG020,4.6,0.0,35.4,179.5,0.0
05BEG021,2.1,0.0,25.3,9.1,24.4
05BEG023,0.0,0.0,25.0,8.8,0.0
05BFG001,1.8,0.0,28.0,0.6,0.0
05BFG003,0.0,0.0,6.4,6.4,0.0
05BFG004,3.0,0.0,0.0,0.0,33.5
05BFG005,4.0,0.0,0.0,0.0,7.6
05BFG006,0.0,9.1,1.5,1.5,0.0


In [17]:
# Completion intervals: AWWID Screens/Perforations cover 5 wells, meta.Production_S all 9.
comp = []
for r in aw['screens'].itertuples():
    comp.append({'station_no': r.station_no, 'type': 'screen', 'source': 'AWWID.Screens',
                 'from_m': round(r.From * FT, 2), 'to_m': round(r.To * FT, 2),
                 'detail': f'slot {r.Slot_Size}"'})
for r in aw['perforations'].itertuples():
    comp.append({'station_no': r.station_no, 'type': 'perforation', 'source': 'AWWID.Perforations',
                 'from_m': round(r.From * FT, 2), 'to_m': round(r.To * FT, 2),
                 'detail': f'dia {r.Diameter}"' if pd.notna(r.Diameter) else None})
for r in c.itertuples():
    lo, _, hi = str(r.production_str).partition(' - ')
    comp.append({'station_no': r.station_no, 'type': 'production (meta)',
                 'source': 'meta.Production_S', 'from_m': float(lo),
                 'to_m': float(hi.split('/')[0]), 'detail': r.aquifer})

cp = pd.DataFrame(comp).merge(c[['station_no', 'code', 'mp_elev_m']], on='station_no')
cp['top_elev_m'] = (cp.mp_elev_m - cp.from_m).round(2)
cp['base_elev_m'] = (cp.mp_elev_m - cp.to_m).round(2)
cp = cp[['station_no', 'code', 'type', 'source', 'from_m', 'to_m',
         'top_elev_m', 'base_elev_m', 'detail']].sort_values(['station_no', 'type'])

# where both exist they should agree -> confirms Production_S for the 4 wells lacking AWWID records
both = cp[cp.type != 'production (meta)'].merge(
    cp[cp.type == 'production (meta)'], on='station_no', suffixes=('', '_meta'))
print("max disagreement AWWID vs Production_S:",
      round(max((both.from_m - both.from_m_meta).abs().max(),
                (both.to_m - both.to_m_meta).abs().max()), 3), "m",
      f"({len(both)} wells with both)")
cp

max disagreement AWWID vs Production_S: 0.04 m (5 wells with both)


,station_no,code,type,source,from_m,to_m,top_elev_m,base_elev_m,detail
13,05BEG018,0760,production (meta),meta.Production_S,59.40,62.50,1256.06,1252.96,Calgary Valley
2,05BEG018,0760,screen,AWWID.Screens,59.44,62.48,1256.02,1252.98,"slot 0.03"""
10,05BEG020,0759,production (meta),meta.Production_S,195.10,201.20,1096.70,1090.60,Calgary Valley
1,05BEG020,0759,screen,AWWID.Screens,195.07,201.17,1096.73,1090.62,"slot 0.03"""
4,05BEG021,0764,perforation,AWWID.Perforations,36.58,54.86,1341.51,1323.23,None
8,05BEG021,0764,production (meta),meta.Production_S,36.60,54.90,1341.49,1323.19,Surficial
9,05BEG023,0364,production (meta),meta.Production_S,21.94,29.56,1262.67,1255.05,Buried Valley
0,05BEG023,0364,screen,AWWID.Screens,21.95,29.57,1262.66,1255.04,"slot 0.025"""
3,05BFG001,0931,perforation,AWWID.Perforations,23.16,29.26,1486.69,1480.59,"dia 0.25"""
12,05BFG001,0931,production (meta),meta.Production_S,23.20,29.30,1486.65,1480.55,Surficial


In [18]:
absval = pd.read_csv(GOWN / 'combined_absval.csv', parse_dates=['date']).set_index('date')
mbtoc = pd.read_csv(GOWN / 'combined_mbtoc.csv', parse_dates=['date']).set_index('date')
codes = c.set_index('code').station_no.to_dict()

wl = absval[list(codes)].rename(columns=codes).dropna(how='all')          # m asl
dtw = mbtoc[list(codes)].rename(columns=codes)                            # m below MP

ws = pd.DataFrame({
    'n_days': wl.notna().sum(),
    'start': wl.apply(lambda s: s.first_valid_index()),
    'end': wl.apply(lambda s: s.last_valid_index()),
    'dtw_mean_m': dtw.mean().round(2),
    'wl_elev_min_m': wl.min().round(2),
    'wl_elev_mean_m': wl.mean().round(2),
    'wl_elev_max_m': wl.max().round(2),
})
ws.index.name = 'station_no'
ws = ws.join(c.set_index('station_no')[['code', 'mp_elev_m']])

# absval + mbtoc should be a constant (the MP elevation); drift means the MP moved
ws['mp_implied_std_m'] = (absval[list(codes)] + mbtoc[list(codes)]).rename(columns=codes).std().round(3)

sw = aw['pump_tests'].groupby('station_no').Static_Water_Level.mean() * FT   # ft -> m below MP
ws['static_wl_at_drilling_m'] = sw.round(2)
ws['static_wl_elev_m'] = (ws.mp_elev_m - ws.static_wl_at_drilling_m).round(2)

ws = ws.reset_index()[['station_no', 'code', 'mp_elev_m', 'n_days', 'start', 'end', 'dtw_mean_m',
                       'wl_elev_min_m', 'wl_elev_mean_m', 'wl_elev_max_m',
                       'static_wl_at_drilling_m', 'static_wl_elev_m', 'mp_implied_std_m']]
ws

,station_no,code,mp_elev_m,n_days,start,end,dtw_mean_m,wl_elev_min_m,wl_elev_mean_m,wl_elev_max_m,static_wl_at_drilling_m,static_wl_elev_m,mp_implied_std_m
0,05BFG006,0301,1603.125,6778,2005-12-12,2024-12-31,2.33,1600.12,1600.79,1602.59,1.48,1601.64,0.000
1,05BFG004,0303,1668.041,6791,2006-05-30,2024-12-31,6.53,1658.77,1661.51,1666.37,5.07,1662.97,0.000
2,05BFG005,0305,2067.549,6937,2005-12-12,2024-12-31,6.83,2058.74,2060.72,2066.97,5.14,2062.41,0.000
3,05BEG021,0764,1378.093,7156,2005-01-01,2024-12-31,27.67,1348.57,1350.42,1357.18,31.60,1346.49,0.000
4,05BEG023,0364,1284.610,7175,2005-01-01,2024-12-31,1.07,1283.27,1283.56,1284.49,NaN,NaN,0.028
5,05BEG020,0759,1291.795,7120,2005-01-01,2024-12-31,2.74,1287.56,1288.73,1291.85,1.52,1290.28,0.558
6,05BFG003,0386,1640.000,6849,2005-12-12,2024-12-31,6.06,1633.55,1633.94,1635.54,5.79,1634.21,0.000
7,05BFG001,0931,1509.849,7113,2005-01-01,2024-12-31,12.07,1496.13,1497.78,1504.08,NaN,NaN,0.000
8,05BEG018,0760,1315.463,7135,2005-01-01,2024-12-31,3.95,1309.70,1311.52,1314.06,2.74,1312.72,0.000


## Regional well logs for the Bow Valley 3-D model

Everything above is the 9 GOWN monitoring wells. This section widens to **every AWWID well inside
the Bow Valley study polygon** and flattens the drillers' logs into stacked intervals suitable for
a 3-D lithology model.

Study area (lon, lat corners, closed clockwise from the NW):

| corner | latitude | longitude |
|---|---|---|
| NW | 51.20845615818 | -115.59501257074348 |
| SW | 50.745153538497334 | -115.60710182485799 |
| SE | 50.73968118154185 | -115.01224273050545 |
| NE | 51.187113688141295 | -114.96769381154475 |

**Collar elevation comes from the 25 m provincial DEM, not from AWWID.** `Wells.Elevation` is null
for 61% of the wells here, and where it does exist it is quoted in feet and disagrees with the DEM
by −1711 to +903 m in the tails (median +5 m, IQR −2 to +14 m). The DEM (EPSG:3400, CGVD28)
resolves at every well and was checked against surveyed top-of-casing at the 9 GOWN wells to
±3 m. AWWID's value is carried as `awwid_elev_m` so the disagreement stays visible.

**Depths are interval bases in feet.** `Lithologies.Depth` is the *bottom* of each interval, so
tops are chained from 0 down the log. Confirmed rather than assumed: the deepest logged interval
equals the report's `Total_Depth_Drilled` exactly for 931 of 942 logs and within 1 ft for the rest.

**One log per well.** 44 wells carry more than one report (mostly 2012 *Old Well-Yield* stubs filed
against 1960s drilling records). The same construction-report rule used above picks the earliest
report that actually records drilling; no well loses its log to this filter.

In [ ]:
from shapely.geometry import Polygon, Point
import rasterio

DEM = Path('/home/py/Documents/alberta_maps/alberta_dem/25m_Raster_GDB_10TM_Resource/25m_Raster.gdb')
BV_POLY = Polygon([(-115.59501257074348, 51.20845615818),      # NW
                   (-115.60710182485799, 50.745153538497334),  # SW
                   (-115.01224273050545, 50.73968118154185),   # SE
                   (-114.96769381154475, 51.187113688141295)]) # NE

lon0, lat0, lon1, lat1 = BV_POLY.bounds
bw = pd.read_sql(
    'SELECT Well_ID, GIC_Well_ID, GOA_Well_Tag_Number, Longitude, Latitude, Elevation,'
    ' Elevation_Obtained, GPS_Obtained, LSD, Section, Township, Range, Meridian'
    ' FROM Wells WHERE Latitude BETWEEN ? AND ? AND Longitude BETWEEN ? AND ?',
    con, params=[lat0, lat1, lon0, lon1])
bw = bw[[BV_POLY.covers(Point(p)) for p in zip(bw.Longitude, bw.Latitude)]].reset_index(drop=True)

# UTM 11N for the model grid; 10TM only to sample the DEM in its own CRS
to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32611', always_xy=True)
to_10tm = Transformer.from_crs('EPSG:4326', 'EPSG:3400', always_xy=True)
bw['x'], bw['y'] = to_utm.transform(bw.Longitude.values, bw.Latitude.values)
ex, nx = to_10tm.transform(bw.Longitude.values, bw.Latitude.values)
with rasterio.open(DEM) as src:
    dem_z = np.array([v[0] for v in src.sample(zip(ex, nx))], dtype='float64')
dem_z[dem_z > 1e30] = np.nan                      # nodata is 3.4e38, never compare by equality

bw['collar_elev'] = dem_z.round(2)
bw['awwid_elev_m'] = (bw.Elevation * FT).round(2).replace(0, np.nan)
bw['elev_discrepancy_m'] = (bw.awwid_elev_m - bw.collar_elev).round(2)

print(f"{len(bw):,} wells inside the polygon   "
      f"DEM null: {bw.collar_elev.isna().sum()}   "
      f"AWWID elevation present: {bw.awwid_elev_m.notna().sum()}")
bw.elev_discrepancy_m.describe(percentiles=[.05, .25, .5, .75, .95]).round(1)

In [ ]:
# Construction report = earliest report that records drilling (same rule as the 9-well summary).
brep = pd.read_sql(
    f'SELECT Well_Report_ID, Well_ID, Type_of_Work, Well_Use, Drilling_Method, Drilling_End_Date,'
    f' Total_Depth_Drilled, Finished_Well_Depth FROM Well_Reports'
    f' WHERE Well_ID IN ({",".join("?" * len(bw))})', con, params=bw.Well_ID.tolist())
brep['_rank'] = brep.Drilling_End_Date.isna().astype(int)
brep = (brep.sort_values(['_rank', 'Drilling_End_Date', 'Well_Report_ID'])
            .groupby('Well_ID').head(1).drop(columns='_rank'))

bl = pd.read_sql(f'SELECT * FROM Lithologies WHERE Well_Report_ID IN'
                 f' ({",".join("?" * len(brep))})', con, params=brep.Well_Report_ID.tolist())
bl = bl[bl.Material != 'Old Well']                # 2012 stub reports, not a real log

# deepest logged interval should reproduce Total_Depth_Drilled
chk = (bl.groupby('Well_Report_ID').Depth.max()
         - brep.set_index('Well_Report_ID').Total_Depth_Drilled).dropna()
print(f"{len(brep):,} construction reports, {len(bl):,} intervals over "
      f"{bl.GIC_Well_ID.nunique():,} wells "
      f"({len(bw) - bl.GIC_Well_ID.nunique():,} wells have no log)")
print(f"deepest interval vs Total_Depth_Drilled: {(chk == 0).sum()}/{len(chk)} exact, "
      f"max |diff| {chk.abs().max():.0f} ft")

In [ ]:
# Material -> class. 68 distinct materials appear in the polygon, so this generalises the CLASS
# dict used for the 9 wells above instead of extending it by hand. Same classes, same rule the
# dict encodes: split on '&', and the FIRST-named material sets the class
# (Clay & Gravel -> fine grained, Gravel & Boulders -> coarse granular, Till & Rocks -> till),
# except that bedrock mixed with drift is flagged 'bedrock (mixed)' (Shale & Gravel).
BEDROCK = ('shale', 'sandstone', 'siltstone', 'claystone', 'mudstone', 'limestone', 'dolomite',
           'coal', 'bedrock', 'conglomerate', 'chert', 'ironstone')
TILL = ('till', 'rocks', 'boulders', 'stones', 'hard pan')
FINE = ('clay', 'silt', 'bentonite', 'muskeg', 'organic')
GRANULAR = ('gravel', 'sand', 'quicksand')


def _part_class(part):
    """Class of one '&'-separated component. Bedrock is tested first because 'sandstone'
    contains 'sand', 'siltstone' contains 'silt' and 'claystone' contains 'clay'."""
    for cls, keys in [('bedrock', BEDROCK), ('till/diamict', TILL),
                      ('fine grained', FINE), ('coarse granular', GRANULAR)]:
        if any(k in part for k in keys):
            return cls
    return 'other'                                    # Topsoil, Fill, Overburden, Unknown


def lith_class(material):
    parts = [_part_class(p.strip()) for p in str(material).lower().split('&')]
    drift = [p for p in parts if p not in ('bedrock', 'other')]
    if 'bedrock' in parts:
        return 'bedrock (mixed)' if drift else 'bedrock'
    return parts[0]


# Depth is the interval BASE, so chain tops from 0 down each log.
bl = bl.sort_values(['Well_Report_ID', 'Depth'], kind='stable')   # some logs are out of sequence
bl['depth_bottom'] = (bl.Depth * FT).round(2)
bl['depth_top'] = (bl.groupby('Well_Report_ID').depth_bottom.shift(fill_value=0.0)).round(2)

# A handful of logs repeat a depth (two materials filed against the same base), which chains to a
# zero-thickness interval. Keep the first of each pair as filed and drop the collapsed one.
zero = bl.depth_bottom <= bl.depth_top
print(f"dropped {zero.sum()} zero-thickness intervals from "
      f"{bl.loc[zero, 'Well_Report_ID'].nunique()} logs (duplicated depth entries)")
bl = bl[~zero]

logs = (bl.rename(columns={'GIC_Well_ID': 'well_id'})
          .merge(bw.rename(columns={'GIC_Well_ID': 'well_id'})[
              ['well_id', 'x', 'y', 'Longitude', 'Latitude', 'collar_elev',
               'awwid_elev_m', 'GOA_Well_Tag_Number']], on='well_id', how='left'))
logs['elev_top'] = (logs.collar_elev - logs.depth_top).round(2)
logs['elev_bottom'] = (logs.collar_elev - logs.depth_bottom).round(2)
logs['thickness_m'] = (logs.depth_bottom - logs.depth_top).round(2)
logs['lith_class'] = logs.Material.map(lith_class)

logs = logs.rename(columns={'Longitude': 'longitude', 'Latitude': 'latitude',
                            'Material': 'material', 'Description': 'description',
                            'Colour': 'colour', 'Water_Bearing': 'water_bearing',
                            'GOA_Well_Tag_Number': 'well_tag', 'Well_Report_ID': 'well_report_id'})
logs = logs[['well_id', 'x', 'y', 'collar_elev', 'depth_top', 'depth_bottom',
             'elev_top', 'elev_bottom', 'thickness_m', 'material', 'lith_class',
             'description', 'colour', 'water_bearing', 'longitude', 'latitude',
             'awwid_elev_m', 'well_tag', 'well_report_id']].reset_index(drop=True)

assert (logs.depth_bottom > logs.depth_top).all(), 'non-positive interval'
assert logs.groupby('well_id').depth_top.min().eq(0).all(), 'a log does not start at ground'
assert logs.collar_elev.notna().all(), 'missing collar elevation'

print(f"{len(logs):,} intervals   {logs.well_id.nunique():,} wells   "
      f"total depth {logs.depth_bottom.max():.1f} m   "
      f"collar range {logs.collar_elev.min():.0f}-{logs.collar_elev.max():.0f} m asl")
logs.head(12)

In [ ]:
print(logs.lith_class.value_counts().to_frame('intervals').assign(
    wells=logs.groupby('lith_class').well_id.nunique(),
    metres=logs.groupby('lith_class').thickness_m.sum().round(0)))

# per-well collar table for the wells that carry a log
collars = (bw.rename(columns={'GIC_Well_ID': 'well_id', 'Longitude': 'longitude',
                              'Latitude': 'latitude', 'GOA_Well_Tag_Number': 'well_tag'})
             [['well_id', 'x', 'y', 'collar_elev', 'longitude', 'latitude',
               'awwid_elev_m', 'elev_discrepancy_m', 'well_tag', 'Elevation_Obtained',
               'LSD', 'Section', 'Township', 'Range', 'Meridian']]
             .merge(logs.groupby('well_id').agg(n_intervals=('material', 'size'),
                                                logged_depth_m=('depth_bottom', 'max')),
                    on='well_id', how='inner')
             .sort_values('well_id').reset_index(drop=True))
collars.columns = [c.lower() for c in collars.columns]
print(f"\n{len(collars):,} logged wells")
collars.head()

### Location quality — read before gridding

Two AWWID artefacts will bite a 3-D interpolation if they go unnoticed, so both are flagged in the
collar table rather than silently dropped.

**The DEM is sound; some coordinates are not.** At the wells located by surveyed GPS (<1 m) the DEM
reproduces AWWID's own elevation to a median of −1.3 m with an sd of 3.9 m. Across every other
location class that sd blows out to 160–280 m. So where the two elevations disagree badly it is the
*well position* that is wrong, not the terrain model — 618 of these wells are logged `Not Verified`
and were placed from the legal land description. `elev_flag` marks wells whose AWWID and DEM
elevations differ by more than 50 m.

**579 wells share a coordinate with another well**, in 160 clusters, one of them 25 wells deep —
LSD-centroid positions rather than distinct locations. `n_at_xy` carries the cluster size. Any
interpolator sees these as coincident control points with conflicting logs, so decide whether to
jitter, average or drop them before gridding.

In [ ]:
gps = pd.read_sql(f'SELECT GIC_Well_ID AS well_id, GPS_Obtained FROM Wells'
                  f' WHERE GIC_Well_ID IN ({",".join("?" * len(collars))})',
                  con, params=collars.well_id.tolist())
collars = collars.merge(gps, on='well_id', how='left').rename(
    columns={'GPS_Obtained': 'gps_obtained'})

collars['n_at_xy'] = collars.groupby(['x', 'y']).well_id.transform('size')
collars['elev_flag'] = collars.elev_discrepancy_m.abs() > 50

# Only the surveyed-GPS wells are positioned well enough to test the DEM against AWWID; the sd
# in every other row is location error, not terrain error.
print(collars.groupby('gps_obtained').elev_discrepancy_m
             .agg(n='count', median='median', sd='std').round(1).sort_values('sd'))

shared = collars[collars.n_at_xy > 1]
print(f"\nelev_flag (|AWWID - DEM| > 50 m): {collars.elev_flag.sum()} wells")
print(f"sharing a coordinate:             {len(shared)} wells in "
      f"{shared.groupby(['x', 'y']).ngroups} clusters, largest {collars.n_at_xy.max()}")

logs = logs.merge(collars[['well_id', 'n_at_xy', 'elev_flag']], on='well_id', how='left')
collars.head()

In [ ]:
OUT3D = DATA / 'bow_valley' / 'regional_3d'
OUT3D.mkdir(parents=True, exist_ok=True)

logs.to_csv(OUT3D / 'well_logs.csv', index=False)
collars.to_csv(OUT3D / 'well_collars.csv', index=False)

print(f"{OUT3D}:")
for f in sorted(OUT3D.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size:>9,} B")